In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# ── Paths ────────────────────────────────────────────────────────────────────
OUTPUT = '../UK_Accident_Analyzed/'
os.makedirs(OUTPUT, exist_ok=True)

def save(fig, name):
    """Save a matplotlib figure to the OUTPUT directory and close it."""
    fig.savefig(f'{OUTPUT}{name}.png', bbox_inches='tight', dpi=120)
    plt.close(fig)

# ── Load cleaned data from cleaning notebook ─────────────────────────────────
df = pd.read_csv('../UK_Accident_cleaned.csv', low_memory=False)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
print(f'Loaded cleaned data: {df.shape}')

---
## Step 3 — Univariate Analysis

Examine the distribution of each key variable in isolation before exploring relationships.  
Variables covered: severity, year, month, day of week, hour, speed limit, road type, area type, casualties, vehicles.

In [60]:
# ── 3a. Severity distribution ─────────────────────────────────────────────────
sev_counts = df['Severity_Label'].value_counts()
sev_pct    = (sev_counts / len(df) * 100).round(2)
print(pd.DataFrame({'count': sev_counts, 'pct': sev_pct}))

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#d62728', '#ff7f0e', '#2ca02c']  # red=Fatal, orange=Serious, green=Slight
sev_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_title('Accident Count by Severity', fontsize=14, fontweight='bold')
ax.set_xlabel('Severity')
ax.set_ylabel('Count')
# Annotate each bar with its value
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + 0.15, p.get_height() + 1000), fontsize=9)
save(fig, '01_severity_distribution')

                 count    pct
Severity_Label               
Slight          794174  85.64
Serious         122236  13.18
Fatal            10977   1.18


In [61]:
# ── 3b. Accidents by Year ─────────────────────────────────────────────────────
year_counts = df.groupby('Year').size()
print(year_counts)

fig, ax = plt.subplots(figsize=(10, 5))
year_counts.plot(kind='line', marker='o', ax=ax, color='steelblue', linewidth=2)
ax.set_title('Total Accidents by Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Accidents')
ax.fill_between(year_counts.index, year_counts.values, alpha=0.15, color='steelblue')
save(fig, '02_accidents_by_year')

Year
2005    128995
2006    121333
2007    115865
2009    102857
2010     99176
2011     95523
2012     90068
2013     82513
2014     91057
dtype: int64


In [62]:
# ── 3c. Fatal accidents by Year ───────────────────────────────────────────────
fatal_year = df[df['Accident_Severity'] == 1].groupby('Year').size()
print(fatal_year)

fig, ax = plt.subplots(figsize=(10, 5))
fatal_year.plot(kind='line', marker='o', ax=ax, color='#d62728', linewidth=2)
ax.set_title('Fatal Accidents by Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Fatal Accidents')
ax.fill_between(fatal_year.index, fatal_year.values, alpha=0.15, color='#d62728')
save(fig, '03_fatalities_by_year')

Year
2005    1727
2006    1724
2007    1585
2009    1226
2010     995
2011    1020
2012     920
2013     845
2014     935
dtype: int64


In [63]:
# ── 3d. Accidents by Month ────────────────────────────────────────────────────
month_order  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_counts = df.groupby('Month_Name').size().reindex(month_order)
print(month_counts)

fig, ax = plt.subplots(figsize=(10, 5))
month_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Accidents by Month', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Accidents')
save(fig, '04_accidents_by_month')

Month_Name
Jan    73184
Feb    67834
Mar    75685
Apr    71328
May    78957
Jun    78555
Jul    81071
Aug    75636
Sep    79778
Oct    84162
Nov    85380
Dec    75817
dtype: int64


In [64]:
# ── 3e. Accidents by Day of Week ──────────────────────────────────────────────
day_order  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df.groupby('Day_Name').size().reindex(day_order)
print(day_counts)

fig, ax = plt.subplots(figsize=(10, 5))
day_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Accidents by Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Day')
ax.set_ylabel('Accidents')
save(fig, '05_accidents_by_day')

Day_Name
Monday       132240
Tuesday      138338
Wednesday    139628
Thursday     140069
Friday       151896
Saturday     123726
Sunday       101490
dtype: int64


In [65]:
# ── 3f. Accidents by Hour of Day ──────────────────────────────────────────────
# Two clear peaks expected: AM rush (~8h) and PM rush (~17h)
hour_counts = df.groupby('Hour').size()
print(hour_counts)

fig, ax = plt.subplots(figsize=(12, 5))
hour_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Accidents by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour')
ax.set_ylabel('Accidents')
save(fig, '06_accidents_by_hour')

Hour
0.0     14449
1.0     10602
2.0      8457
3.0      6767
4.0      5326
5.0      7471
6.0     15814
7.0     37942
8.0     67199
9.0     46270
10.0    40960
11.0    47004
12.0    54057
13.0    55973
14.0    55630
15.0    71412
16.0    74323
17.0    81605
18.0    65738
19.0    49228
20.0    36484
21.0    29069
22.0    25339
23.0    20258
dtype: int64


In [66]:
# ── 3g. Speed limit distribution ──────────────────────────────────────────────
speed_counts = df['Speed_limit'].value_counts().sort_index()
print(speed_counts)

fig, ax = plt.subplots(figsize=(9, 5))
speed_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Accidents by Speed Limit', fontsize=14, fontweight='bold')
ax.set_xlabel('Speed Limit (mph)')
ax.set_ylabel('Accidents')
save(fig, '07_speed_limit_dist')

# ── 3h. Road Type distribution ────────────────────────────────────────────────
df['Road_Type_Label'] = df['Road_Type'].fillna('Unknown')
rt_counts = df['Road_Type_Label'].value_counts()
print(rt_counts)

# ── 3i. Urban vs Rural ────────────────────────────────────────────────────────
area_counts = df['Area_Type'].value_counts()
print(area_counts)

# ── 3j. Casualties & Vehicle count descriptive stats ─────────────────────────
print(df['Number_of_Casualties'].describe())
print(f"Accidents with 5+ casualties: {(df['Number_of_Casualties'] >= 5).sum()}")

print(df['Number_of_Vehicles'].describe())

Speed_limit
10.0        12
15.0        10
20.0      9505
30.0    631953
40.0     72138
50.0     27343
60.0    127606
70.0     58820
Name: count, dtype: int64

Road_Type_Label
Single carriageway    694990
Dual carriageway      137681
Roundabout             60765
One way street         19783
Slip road               9073
Unknown                 5095
Name: count, dtype: int64

Area_Type
Urban          643970
Rural          283342
Unallocated        75
Name: count, dtype: int64

count    927387.000000
mean          1.343509
std           0.820306
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          93.000000
Name: Number_of_Casualties, dtype: float64
Accidents with 5+ casualties: 9113

count    927387.000000
mean          1.828089
std           0.701695
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max          32.000000
Name: Number_of_Vehicles, dtype: float64


---
## Step 4 — Bivariate Analysis: Severity Drivers

For each environmental/contextual factor we compute:
- **fatal_rate** = (fatal accidents / total accidents) × 100
- **serious_rate** = (serious accidents / total) × 100

This normalises for volume differences between categories (e.g., far more 30mph accidents than 70mph).

In [67]:
def fatal_rate_by(col, label_map=None, top_n=None):
    """
    Compute fatal and serious rate per unique value in `col`.

    Parameters
    ----------
    col       : str   — column to group by
    label_map : dict  — optional mapping from raw code → readable label
    top_n     : int   — if set, return only the top N categories by total volume

    Returns
    -------
    DataFrame with columns: total, fatal, serious, fatal_rate, serious_rate
    """
    grp = df.groupby(col)['Accident_Severity'].agg(
        total  ='count',
        fatal  =lambda x: (x == 1).sum(),
        serious=lambda x: (x == 2).sum()
    )
    grp['fatal_rate']   = (grp['fatal']   / grp['total'] * 100).round(2)
    grp['serious_rate'] = (grp['serious'] / grp['total'] * 100).round(2)
    if label_map:
        grp.index = grp.index.map(label_map)
    if top_n:
        grp = grp.nlargest(top_n, 'total')
    return grp

In [68]:
# ── 4a. Severity × Speed Limit ────────────────────────────────────────────────
# Higher speed limits → higher kinetic energy at impact → higher fatality rate
speed_sev = fatal_rate_by('Speed_limit')
print(speed_sev)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
speed_sev['fatal_rate'].plot(kind='bar', ax=axes[0], color='#d62728', edgecolor='black')
axes[0].set_title('Fatal Rate by Speed Limit', fontweight='bold')
axes[0].set_xlabel('Speed Limit (mph)')
axes[0].set_ylabel('Fatal Rate (%)')
speed_sev['total'].plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Total Accidents by Speed Limit', fontweight='bold')
axes[1].set_xlabel('Speed Limit (mph)')
axes[1].set_ylabel('Total Accidents')
plt.tight_layout()
save(fig, '08_severity_speed_limit')

              total  fatal  serious  fatal_rate  serious_rate
Speed_limit                                                  
10.0             12      2        1       16.67          8.33
15.0             10      0        0        0.00          0.00
20.0           9505     57     1343        0.60         14.13
30.0         631953   4260    77640        0.67         12.29
40.0          72138    999     9611        1.38         13.32
50.0          27343    558     4054        2.04         14.83
60.0         127606   3739    22715        2.93         17.80
70.0          58820   1362     6872        2.32         11.68


In [69]:
# ── 4b. Severity × Light Conditions ──────────────────────────────────────────
light_sev = fatal_rate_by('Light_Conditions')
print(light_sev)

fig, ax = plt.subplots(figsize=(10, 5))
light_sev['fatal_rate'].plot(kind='bar', ax=ax, color='#d62728', edgecolor='black')
ax.set_title('Fatal Rate by Light Conditions', fontsize=14, fontweight='bold')
ax.set_xlabel('Light Condition')
ax.set_ylabel('Fatal Rate (%)')
plt.xticks(rotation=30, ha='right')
save(fig, '09_severity_light_conditions')

                                            total  fatal  serious  fatal_rate  \
Light_Conditions                                                                
Darkeness: No street lighting               43061   1736     8082        4.03   
Darkness: Street lighting unknown            7175     64      875        0.89   
Darkness: Street lights present and lit    197422   2665    29024        1.35   
Darkness: Street lights present but unlit    3737     65      558        1.74   
Daylight: Street light present             675992   6447    83697        0.95   

                                           serious_rate  
Light_Conditions                                         
Darkeness: No street lighting                     18.77  
Darkness: Street lighting unknown                 12.20  
Darkness: Street lights present and lit           14.70  
Darkness: Street lights present but unlit         14.93  
Daylight: Street light present                    12.38  


In [70]:
# ── 4c. Severity × Weather Conditions ────────────────────────────────────────
weather_sev = fatal_rate_by('Weather_Conditions')
print(weather_sev)

fig, ax = plt.subplots(figsize=(12, 5))
weather_sev['fatal_rate'].dropna().plot(kind='bar', ax=ax, color='#d62728', edgecolor='black')
ax.set_title('Fatal Rate by Weather Conditions', fontsize=14, fontweight='bold')
ax.set_xlabel('Weather')
ax.set_ylabel('Fatal Rate (%)')
plt.xticks(rotation=30, ha='right')
save(fig, '10_severity_weather')

                             total  fatal  serious  fatal_rate  serious_rate
Weather_Conditions                                                          
Fine with high winds         10439    189     1538        1.81         14.73
Fine without high winds     744846   9139   100830        1.23         13.54
Fog or mist                   4717     91      677        1.93         14.35
Other                        21582    187     2294        0.87         10.63
Raining with high winds      12066    148     1568        1.23         13.00
Raining without high winds  109994   1047    13061        0.95         11.87
Snowing with high winds       1134      6      127        0.53         11.20
Snowing without high winds    6750     43      637        0.64          9.44


In [71]:
# ── 4d. Severity × Road Surface ───────────────────────────────────────────────
surface_sev = fatal_rate_by('Road_Surface_Conditions')
print(surface_sev)

fig, ax = plt.subplots(figsize=(10, 5))
surface_sev['fatal_rate'].dropna().plot(kind='bar', ax=ax, color='#d62728', edgecolor='black')
ax.set_title('Fatal Rate by Road Surface', fontsize=14, fontweight='bold')
ax.set_xlabel('Road Surface')
ax.set_ylabel('Fatal Rate (%)')
plt.xticks(rotation=30, ha='right')
save(fig, '11_severity_road_surface')

                            total  fatal  serious  fatal_rate  serious_rate
Road_Surface_Conditions                                                    
Dry                        644227   7399    86440        1.15         13.42
Flood (Over 3cm of water)    1145     21      148        1.83         12.93
Frost/Ice                   18544    177     2088        0.95         11.26
Normal                       1649     10      169        0.61         10.25
Snow                         5933     42      552        0.71          9.30
Wet/Damp                   255889   3328    32839        1.30         12.83


In [72]:
# ── 4e. Severity × Road Type ──────────────────────────────────────────────────
roadtype_sev = df.groupby('Road_Type_Label')['Accident_Severity'].agg(
    total='count',
    fatal=lambda x: (x == 1).sum()
)
roadtype_sev['fatal_rate'] = (roadtype_sev['fatal'] / roadtype_sev['total'] * 100).round(2)
print(roadtype_sev)

fig, ax = plt.subplots(figsize=(10, 5))
roadtype_sev['fatal_rate'].sort_values(ascending=False).plot(kind='bar', ax=ax, color='#d62728', edgecolor='black')
ax.set_title('Fatal Rate by Road Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Road Type')
ax.set_ylabel('Fatal Rate (%)')
plt.xticks(rotation=30, ha='right')
save(fig, '12_severity_road_type')

                     total  fatal  fatal_rate
Road_Type_Label                              
Dual carriageway    137681   2264        1.64
One way street       19783    126        0.64
Roundabout           60765    183        0.30
Single carriageway  694990   8297        1.19
Slip road             9073     66        0.73
Unknown               5095     41        0.80


In [73]:
# ── 4f. Severity × Urban vs Rural ─────────────────────────────────────────────
# Rural roads have higher fatal rates despite lower total volumes
area_sev = fatal_rate_by('Area_Type')
print(area_sev)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
area_sev['fatal_rate'].plot(kind='bar', ax=axes[0], color='#d62728', edgecolor='black')
axes[0].set_title('Fatal Rate: Urban vs Rural', fontweight='bold')
axes[0].set_ylabel('Fatal Rate (%)')
area_sev['total'].plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Total Accidents: Urban vs Rural', fontweight='bold')
axes[1].set_ylabel('Total Accidents')
plt.tight_layout()
save(fig, '13_severity_urban_rural')

              total  fatal  serious  fatal_rate  serious_rate
Area_Type                                                    
Rural        283342   6303    43938        2.22         15.51
Unallocated      75      0        6        0.00          8.00
Urban        643970   4674    78292        0.73         12.16


In [74]:
# ── 4g. Severity × Hour ───────────────────────────────────────────────────────
hour_sev = df.groupby(['Hour', 'Severity_Label']).size().unstack(fill_value=0)
print(hour_sev)

fig, ax = plt.subplots(figsize=(14, 5))
hour_sev[['Fatal', 'Serious', 'Slight']].plot(
    kind='bar', stacked=True, ax=ax,
    color=['#d62728', '#ff7f0e', '#2ca02c'], edgecolor='none'
)
ax.set_title('Accidents by Hour and Severity', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour')
ax.set_ylabel('Accidents')
save(fig, '14_severity_by_hour')

# Fatal rate separately to separate volume effect from rate effect
hour_fatal_rate = df.groupby('Hour')['Accident_Severity'].agg(
    total='count',
    fatal=lambda x: (x == 1).sum()
)
hour_fatal_rate['fatal_rate'] = (hour_fatal_rate['fatal'] / hour_fatal_rate['total'] * 100).round(2)
print('\nFatal rate by hour (top 10):')
print(hour_fatal_rate.sort_values('fatal_rate', ascending=False).head(10))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hour_fatal_rate.index, hour_fatal_rate['fatal_rate'], marker='o', color='#d62728', linewidth=2)
ax.set_title('Fatal Rate by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour')
ax.set_ylabel('Fatal Rate (%)')
ax.fill_between(hour_fatal_rate.index, hour_fatal_rate['fatal_rate'], alpha=0.15, color='#d62728')
save(fig, '15_fatal_rate_by_hour')

Severity_Label  Fatal  Serious  Slight
Hour                                  
0.0               410     2688   11351
1.0               313     2047    8242
2.0               295     1662    6500
3.0               228     1337    5202
4.0               195     1007    4124
5.0               227     1344    5900
6.0               273     2433   13108
7.0               400     4974   32568
8.0               418     7085   59696
9.0               419     4897   40954
10.0              511     4919   35530
11.0              495     5522   40987
12.0              487     6389   47181
13.0              551     6709   48713
14.0              561     7019   48050
15.0              608     9177   61627
16.0              700     9753   63870
17.0              742    10371   70492
18.0              621     8645   56472
19.0              606     6771   41851
20.0              494     5390   30600
21.0              491     4501   24077
22.0              494     4163   20682
23.0              438    

In [75]:
# ── 4h. Severity × Day of Week ────────────────────────────────────────────────
day_sev = df.groupby(['Day_Name', 'Severity_Label']).size().unstack(fill_value=0)
day_sev = day_sev.reindex(day_order)
print(day_sev)

fig, ax = plt.subplots(figsize=(12, 5))
day_sev[['Fatal', 'Serious', 'Slight']].plot(
    kind='bar', stacked=True, ax=ax,
    color=['#d62728', '#ff7f0e', '#2ca02c'], edgecolor='none'
)
ax.set_title('Accidents by Day of Week and Severity', fontsize=14, fontweight='bold')
save(fig, '16_severity_by_day')

Severity_Label  Fatal  Serious  Slight
Day_Name                              
Monday           1455    16734  114051
Tuesday          1426    17284  119628
Wednesday        1352    17313  120963
Thursday         1454    17776  120839
Friday           1651    19696  130549
Saturday         1887    17864  103975
Sunday           1752    15569   84169


In [76]:
# ── 4i. Severity × Month (dual-axis: volume + fatal rate) ─────────────────────
month_sev_rate = df.groupby('Month_Name')['Accident_Severity'].agg(
    total='count',
    fatal=lambda x: (x == 1).sum()
)
month_sev_rate['fatal_rate'] = (month_sev_rate['fatal'] / month_sev_rate['total'] * 100).round(2)
month_sev_rate = month_sev_rate.reindex(month_order)
print(month_sev_rate)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(month_order, month_sev_rate['total'], color='steelblue', label='Total', alpha=0.7)
ax2 = ax.twinx()  # secondary y-axis for fatal rate line
ax2.plot(month_order, month_sev_rate['fatal_rate'], marker='o', color='#d62728', linewidth=2, label='Fatal Rate')
ax.set_title('Monthly Accidents vs Fatal Rate', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Accidents')
ax2.set_ylabel('Fatal Rate (%)', color='#d62728')
save(fig, '17_monthly_severity')

            total  fatal  fatal_rate
Month_Name                          
Jan         73184    890        1.22
Feb         67834    805        1.19
Mar         75685    900        1.19
Apr         71328    870        1.22
May         78957    865        1.10
Jun         78555    912        1.16
Jul         81071    897        1.11
Aug         75636    988        1.31
Sep         79778    954        1.20
Oct         84162    977        1.16
Nov         85380    961        1.13
Dec         75817    958        1.26


In [77]:
# ── 4j. Fatality rate trend — Year over Year ──────────────────────────────────
year_fatal_rate = df.groupby('Year')['Accident_Severity'].agg(
    total='count',
    fatal=lambda x: (x == 1).sum()
)
year_fatal_rate['fatal_rate'] = (year_fatal_rate['fatal'] / year_fatal_rate['total'] * 100).round(2)
print(year_fatal_rate)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(year_fatal_rate.index, year_fatal_rate['total'], color='steelblue', label='Total', alpha=0.7)
ax2 = ax.twinx()
ax2.plot(year_fatal_rate.index, year_fatal_rate['fatal_rate'], marker='o', color='#d62728', linewidth=2)
ax.set_title('Total Accidents vs Fatal Rate by Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Accidents')
ax2.set_ylabel('Fatal Rate (%)', color='#d62728')
save(fig, '18_fatal_rate_by_year')

       total  fatal  fatal_rate
Year                           
2005  128995   1727        1.34
2006  121333   1724        1.42
2007  115865   1585        1.37
2009  102857   1226        1.19
2010   99176    995        1.00
2011   95523   1020        1.07
2012   90068    920        1.02
2013   82513    845        1.02
2014   91057    935        1.03


In [78]:
# ── 4k. Average casualties by speed limit ─────────────────────────────────────
cas_speed = df.groupby('Speed_limit')['Number_of_Casualties'].mean().round(3)
print(cas_speed)

# ── 4l. Average vehicles involved by severity ─────────────────────────────────
veh_sev = df.groupby('Severity_Label')['Number_of_Vehicles'].mean().round(3)
print(veh_sev)

# ── 4m. Top 15 districts by fatal accident count ──────────────────────────────
district_fatal = df[df['Accident_Severity'] == 1].groupby('Local_Authority_(District)').size()
print(district_fatal.nlargest(15))

Speed_limit
10.0    1.167
15.0    1.100
20.0    1.185
30.0    1.268
40.0    1.443
50.0    1.509
60.0    1.525
70.0    1.586
Name: Number_of_Casualties, dtype: float64

Severity_Label
Fatal      1.757
Serious    1.671
Slight     1.853
Name: Number_of_Vehicles, dtype: float64

Local_Authority_(District)
300    235
927    181
231    167
753    139
596    132
102    127
926    124
169    107
938    107
211    103
351    102
479     98
285     96
91      95
286     95
dtype: int64


---
## Step 5 — Temporal Heatmaps

Cross-tabulate **Hour** against **Day of Week** to reveal accident hotspot windows.  
Two heatmaps: all accidents (frequency) and fatal-only accidents (severity focus).

In [79]:
# ── 5a. All accidents: Hour × Day ─────────────────────────────────────────────
heat_all = df.groupby(['Hour', 'Day_Name']).size().unstack(fill_value=0)
heat_all = heat_all[day_order]  # reorder columns Mon→Sun

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(heat_all, cmap='YlOrRd', ax=ax, fmt=',d', annot=False, linewidths=0.3)
ax.set_title('Accident Frequency: Hour × Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Day')
ax.set_ylabel('Hour')
save(fig, '19_heatmap_hour_day_all')

# ── 5b. Fatal-only: Hour × Day ────────────────────────────────────────────────
heat_fatal = df[df['Accident_Severity'] == 1].groupby(['Hour', 'Day_Name']).size().unstack(fill_value=0)
heat_fatal = heat_fatal.reindex(columns=day_order, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(heat_fatal, cmap='Reds', ax=ax, annot=False, linewidths=0.3)
ax.set_title('FATAL Accident Frequency: Hour × Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Day')
ax.set_ylabel('Hour')
save(fig, '20_heatmap_hour_day_fatal')

---
## Step 6 — Multivariate Analysis

Examine **interactions between two factors simultaneously** on fatal rate.  
All pivots filtered to cells with >100 records to avoid noisy small-sample rates.

In [80]:
# Binary flag for fatal outcome — used throughout multivariate section
df['Is_Fatal'] = (df['Accident_Severity'] == 1).astype(int)

# ── Simplify light condition labels for readability in pivot tables ────────────
df['Light_Simple'] = df['Light_Conditions'].map({
    'Daylight: Street light present'           : 'Daylight',
    'Darkness: Street lights present and lit'  : 'Dark-lit',
    'Darkness: Street lights present but unlit': 'Dark-unlit',
    'Darkeness: No street lighting'            : 'Dark-no light',
    'Darkness: Street lighting unknown'        : 'Dark-unknown'
})

In [81]:
# ── 6a. Speed × Light × Severity ─────────────────────────────────────────────
# Does darkness amplify the fatality risk at high speed?
speed_light = df.groupby(['Speed_limit', 'Light_Simple'])['Is_Fatal'].agg(['mean', 'count']).reset_index()
speed_light['fatal_rate'] = (speed_light['mean'] * 100).round(2)
speed_light = speed_light[speed_light['count'] > 100]  # filter low-volume cells
pivot_sl = speed_light.pivot(index='Speed_limit', columns='Light_Simple', values='fatal_rate')
print(pivot_sl)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot_sl, cmap='Reds', ax=ax, annot=True, fmt='.1f', linewidths=0.5)
ax.set_title('Fatal Rate (%): Speed Limit × Light Conditions', fontsize=14, fontweight='bold')
save(fig, '21_multivar_speed_light_severity')

Light_Simple  Dark-lit  Dark-no light  Dark-unknown  Dark-unlit  Daylight
Speed_limit                                                              
20.0              0.75            NaN           NaN         NaN      0.56
30.0              1.08           1.67          0.52        1.00      0.53
40.0              2.34           2.95          1.35        3.01      0.99
50.0              2.23           3.89          1.87        6.06      1.75
60.0              2.66           4.21          2.83        2.69      2.59
70.0              2.80           5.25          1.13        3.23      1.69


In [82]:
# ── 6b. Road Type × Urban/Rural × Fatal rate ──────────────────────────────────
rt_area = df.groupby(['Road_Type_Label', 'Area_Type'])['Is_Fatal'].agg(['mean', 'count']).reset_index()
rt_area['fatal_rate'] = (rt_area['mean'] * 100).round(2)
rt_area = rt_area[rt_area['count'] > 50]
pivot_rt = rt_area.pivot(index='Road_Type_Label', columns='Area_Type', values='fatal_rate')
print(pivot_rt)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot_rt, cmap='Reds', ax=ax, annot=True, fmt='.1f', linewidths=0.5)
ax.set_title('Fatal Rate (%): Road Type × Urban/Rural', fontsize=14, fontweight='bold')
save(fig, '22_multivar_roadtype_area_severity')

Area_Type           Rural  Unallocated  Urban
Road_Type_Label                              
Dual carriageway     2.37          NaN   1.13
One way street       0.85          NaN   0.62
Roundabout           0.37          NaN   0.27
Single carriageway   2.42          0.0   0.70
Slip road            0.85          NaN   0.58
Unknown              1.41          NaN   0.57


In [83]:
# ── 6c. Weather × Road Surface × Severity ────────────────────────────────────
# Simplify verbose weather and surface labels to shorter tokens
df['Weather_Simple'] = df['Weather_Conditions'].map({
    'Fine without high winds' : 'Fine',
    'Fine with high winds'    : 'Fine+Wind',
    'Raining without high winds': 'Rain',
    'Raining with high winds' : 'Rain+Wind',
    'Snowing without high winds': 'Snow',
    'Snowing with high winds' : 'Snow+Wind',
    'Fog or mist'             : 'Fog',
    'Other'                   : 'Other',
    'Unknown'                 : 'Unknown'
})
df['Surface_Simple'] = df['Road_Surface_Conditions'].map({
    'Dry'                          : 'Dry',
    'Normal'                       : 'Normal',
    'Wet/Damp'                     : 'Wet',
    'Snow'                         : 'Snow',
    'Frost/Ice'                    : 'Ice',
    'Flood (Over 3cm of water)'    : 'Flood'
})

wx_surf = df.groupby(['Weather_Simple', 'Surface_Simple'])['Is_Fatal'].agg(['mean', 'count']).reset_index()
wx_surf['fatal_rate'] = (wx_surf['mean'] * 100).round(2)
wx_surf = wx_surf[wx_surf['count'] > 100]
pivot_wx = wx_surf.pivot(index='Weather_Simple', columns='Surface_Simple', values='fatal_rate')
print(pivot_wx)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot_wx, cmap='Reds', ax=ax, annot=True, fmt='.1f', linewidths=0.5)
ax.set_title('Fatal Rate (%): Weather × Road Surface', fontsize=14, fontweight='bold')
save(fig, '23_multivar_weather_surface_severity')

Surface_Simple   Dry  Flood   Ice  Normal  Snow   Wet
Weather_Simple                                       
Fine            1.15   1.47  1.06     0.0  0.93  1.65
Fine+Wind       1.84    NaN  0.99     NaN   NaN  1.80
Fog             3.00    NaN  1.73     NaN   NaN  1.74
Other           0.57    NaN  0.73     NaN  0.54  1.01
Rain            0.39   1.76  0.87     NaN  2.80  0.95
Rain+Wind        NaN   1.37   NaN     NaN   NaN  1.22
Snow             NaN    NaN  0.47     NaN  0.59  0.73
Snow+Wind        NaN    NaN  0.00     NaN  0.68  0.49


In [84]:
# ── 6d. Time Period × Speed Limit × Fatal Rate ────────────────────────────────
# Bin hours into named time periods for readability
df['Time_Period'] = pd.cut(
    df['Hour'],
    bins=[-1, 6, 9, 12, 16, 19, 22, 24],
    labels=['Night(0-6)', 'AM Rush(7-9)', 'Midday(10-12)',
            'Afternoon(13-16)', 'PM Rush(17-19)', 'Evening(20-22)', 'Late Night(23-24)']
)

hour_speed = df.groupby(['Time_Period', 'Speed_limit'])['Is_Fatal'].agg(['mean', 'count']).reset_index()
hour_speed['fatal_rate'] = (hour_speed['mean'] * 100).round(2)
hour_speed = hour_speed[hour_speed['count'] > 100]
pivot_hs = hour_speed.pivot(index='Time_Period', columns='Speed_limit', values='fatal_rate')
print(pivot_hs)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot_hs, cmap='Reds', ax=ax, annot=True, fmt='.1f', linewidths=0.5)
ax.set_title('Fatal Rate (%): Time Period × Speed Limit', fontsize=14, fontweight='bold')
save(fig, '24_multivar_timeperiod_speed_severity')

Speed_limit        20.0  30.0  40.0  50.0  60.0  70.0
Time_Period                                          
Night(0-6)         1.47  1.75  3.70  3.76  4.83  5.57
AM Rush(7-9)       0.73  0.48  0.88  0.98  1.90  1.53
Midday(10-12)      0.43  0.64  0.83  2.12  2.63  1.72
Afternoon(13-16)   0.59  0.50  1.03  1.78  2.84  1.55
PM Rush(17-19)     0.41  0.56  1.18  1.86  2.86  1.67
Evening(20-22)     0.72  0.91  2.28  2.94  3.75  3.88
Late Night(23-24)  0.00  1.29  3.18  3.93  4.20  5.44


In [85]:
# ── 6e. Weekday vs Weekend × Speed × Fatal Rate ───────────────────────────────
df['Day_Type'] = df['Day_Name'].apply(lambda x: 'Weekend' if x in ['Saturday', 'Sunday'] else 'Weekday')
wknd_speed = df.groupby(['Day_Type', 'Speed_limit'])['Is_Fatal'].agg(['mean', 'count']).reset_index()
wknd_speed['fatal_rate'] = (wknd_speed['mean'] * 100).round(2)
wknd_speed = wknd_speed[wknd_speed['count'] > 100]
pivot_ws = wknd_speed.pivot(index='Day_Type', columns='Speed_limit', values='fatal_rate')
print(pivot_ws)

Speed_limit  20.0  30.0  40.0  50.0  60.0  70.0
Day_Type                                       
Weekday      0.51  0.62  1.22  1.76  2.57  2.04
Weekend      0.90  0.85  1.91  2.88  3.88  3.14
